In [ ]:
# Classifier trained on LHS results

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

def classify(val):
    if val <=  225:
        return 0
    if val <= 550:
        return 1
    if val <= 775:
        return 2
    return 3

# TRAINING ON LHS
# f = "/project/jozik/ncollier/repos/varmodel3_emews/results/lhs_8K_0728_11042024_results.csv"
# df = pd.read_csv(f)
# df['group'] = df.runtime.apply(classify)
# df.value_counts('group')
# df = df[['biting_rate', 'immunity_loss_rate', 'switching_rate_BC', 'ectopic_recombination_rate_BC', 
#          'immigration_rate_fraction', 'n_genes_initial', 'switching_rate_A', 'ectopic_recombination_rate_A',
#          'var_groups_ratio_regional_pool_A', 'group']]

# TRAINING ON RUNTIME TESTS

off = False
if off:
    f = "/project/jozik/ncollier/repos/varmodel3/emews/experiments/runtime_tests_gi_off_032025/runtime_results.csv"
    cols = ['biting_rate', 'immunity_loss_rate', 'switching_rate_BC', 'ectopic_recombination_rate_BC', 
            'immigration_rate_fraction', 'n_genes_initial', 'switching_rate_A', 'coinfection_reduces_transmission_exponential_decay_param',
            'var_groups_ratio_regional_pool_A', 'group']
else:
    f = "/project/jozik/ncollier/repos/varmodel3/emews/experiments/runtime_tests_gi_on_032025/runtime_results.csv"
    cols = ['biting_rate', 'immunity_loss_rate', 'switching_rate_BC', 'ectopic_recombination_rate_BC', 
            'immigration_rate_fraction', 'n_genes_initial', 'switching_rate_A', 'coinfection_reduces_transmission_exponential_decay_param',
            'var_groups_ratio_regional_pool_A', "generalized_immunity_transmissibility_param",
            "generalized_immunity_detectability_param", 'group']
    
df  = pd.read_csv(f)
df['group'] = df.runtime.apply(classify)
df.value_counts('group')
df = df[cols]

X = df.drop('group', axis=1).values
y = df['group'].values

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

rf = RandomForestClassifier()
# rf.fit(X_train, y_train)
# y_pred = rf.predict(X_test)

# accuracy = accuracy_score(y_test, y_pred)
# print("Accuracy:", accuracy)

param_dist = {'n_estimators': randint(50, 500),
              'max_depth': randint(1, 20)}

# Create a random forest classifier
rf = RandomForestClassifier()

# Use random search to find the best hyperparameters
rand_search = RandomizedSearchCV(rf, 
                                 param_distributions=param_dist, 
                                 n_iter=20, 
                                 cv=5)

# Fit the random search object to the data
rand_search.fit(X_train, y_train)

# Create a variable for the best model
best_rf = rand_search.best_estimator_

# Print the best hyperparameters
print('Best hyperparameters:',  rand_search.best_params_)
y_pred = best_rf.predict(X_test)
accuracy_score(y_test, y_pred)


In [ ]:
# save the classifier
import pickle

f = "gi_on_032025/rf_model_gi_on_032025.pkl"
with open(f, "wb") as fout:
    pickle.dump(best_rf, fout)

with open(f, 'rb') as fin:
    rf = pickle.load(fin)

y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)


In [ ]:
# use the classifier 'rf' to determine buckets
def update_task_type(rf, biting_rate,
                     immunity_loss_rate, switching_rate_BC, ectopic_recombination_rate_BC,
                     immigration_rate_fraction,
                     n_genes_initial, switching_rate_A, ectopic_recombination_rate_A,
                     var_groups_ratio_regional_pool_A):
    # if biting_rate <=  3.06E-05:
    #     task_type = 14
    # elif biting_rate <=  5.38e-05:
    #     task_type = 15
    # elif  biting_rate <= 7.69e-05:
    #     task_type = 16
    # elif biting_rate <= 0.0001:
    #     task_type = 17

    group = rf.predict(np.array([[biting_rate, immunity_loss_rate, switching_rate_BC, ectopic_recombination_rate_BC,
                                  immigration_rate_fraction, n_genes_initial, switching_rate_A,
                                  ectopic_recombination_rate_A, var_groups_ratio_regional_pool_A]]))
